# 🇳🇵 Nepali Political Sentiment — Classical ML Notebook

Dataset: pre-cleaned CSV with `text` (Devanagari) and `label` (1 = Positive, 2 = Negative) columns.

Pipeline:
1. Load & inspect data
2. Handle class imbalance via upsampling
3. Vectorise with **TF-IDF** (primary) and **Count Vectorizer** (secondary)
4. Train & evaluate 7 classifiers: Logistic Regression, SVM, Random Forest, Decision Tree, Naive Bayes, XGBoost, AdaBoost
5. Compare all models in a summary table
6. Save the best model

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings, os, json
matplotlib.use('Agg')
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed — run: pip install xgboost")

import joblib
print("✅ All imports successful")


✅ All imports successful


## 2. Load & Inspect Dataset

Update `DATA_PATH` to point to your CSV file.

In [2]:
# ─── CONFIG ─────────────────────────────────────────────────────
DATA_PATH   = 'cleaned_dataset_3label.csv'   # <-- change this
TEXT_COL    = 'text'
LABEL_COL   = 'label'
RANDOM_SEED = 42
TEST_SIZE   = 0.2
# ────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)

# Keep only text & label; drop rows with missing values
df = df[[TEXT_COL, LABEL_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].apply(lambda x: len(x.split()) > 1)]   # drop 1-word rows
df = df.drop_duplicates(subset=[TEXT_COL])

print(f"Total samples after cleaning: {len(df)}")
print(f"\nLabel distribution:\n{df[LABEL_COL].value_counts()}")
df.head(5)


Total samples after cleaning: 75956

Label distribution:
label
1.0    44911
2.0    26803
0.0     4242
Name: count, dtype: int64


,text,label
0,हर्कराज साम्पाङ बालेन शाहहरुको दगुरेको गाउँपाल...,2.0
1,जनप्रतिनिधि सर्वोच्च न्यायाधीश महाभियोग हटाउनु,2.0
2,च्याखे थापेर करारका जनप्रतिनिधिने अत्याचार निज...,2.0
3,सत्तारुढ गठबन्धनका नेताहरू रविलाई सरकारमा नल्य...,2.0
4,क्षेत्रमा खराब ब्यक्तिहरु छैनन खराब राजनीति दल...,2.0


## 3. Remap Labels → 0 / 1

Original labels are `1` (Positive) and `2` (Negative). Remap to `0` and `1` so all sklearn metrics work without extra arguments.

In [4]:
# 1 -> 0 (Positive),  2 -> 1 (Negative)
label_map = {1: 0, 2: 1}
df['label_enc'] = df[LABEL_COL].map(label_map)

print("Remapped label counts:")
print(df['label_enc'].value_counts())
print("\n0 = Positive  |  1 = Negative")


Remapped label counts:
label_enc
0    44954
1    26807
Name: count, dtype: int64

0 = Positive  |  1 = Negative


## 4. Handle Class Imbalance — Upsample Minority Class

Match the approach from the original notebook: duplicate minority-class rows until both classes are equal in size.

In [5]:
pos_df = df[df['label_enc'] == 0]
neg_df = df[df['label_enc'] == 1]

print(f"Before upsampling → Positive: {len(pos_df)}, Negative: {len(neg_df)}")

majority_n = max(len(pos_df), len(neg_df))

if len(pos_df) < len(neg_df):
    extra = pos_df.sample(n=majority_n - len(pos_df), replace=True, random_state=RANDOM_SEED)
    balanced_df = pd.concat([pos_df, extra, neg_df]).reset_index(drop=True)
else:
    extra = neg_df.sample(n=majority_n - len(neg_df), replace=True, random_state=RANDOM_SEED)
    balanced_df = pd.concat([pos_df, neg_df, extra]).reset_index(drop=True)

balanced_df = balanced_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"After  upsampling → {balanced_df['label_enc'].value_counts().to_dict()}")
print(f"Total samples     : {len(balanced_df)}")


Before upsampling → Positive: 44954, Negative: 26807
After  upsampling → {1: 44954, 0: 44954}
Total samples     : 89908


## 5. Train / Test Split

In [6]:
X = balanced_df[TEXT_COL]
y = balanced_df['label_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

print(f"Train: {len(X_train)} samples   |   Test: {len(X_test)} samples")
print(f"Train label dist: {y_train.value_counts().to_dict()}")
print(f"Test  label dist: {y_test.value_counts().to_dict()}")


Train: 71926 samples   |   Test: 17982 samples
Train label dist: {0: 35963, 1: 35963}
Test  label dist: {1: 8991, 0: 8991}


## 6. TF-IDF Vectorisation

`sublinear_tf=True` compresses very high term frequencies. `ngram_range=(1,2)` captures both single words and bigrams like "भ्रष्टाचार गर्ने".

In [25]:
# Section 6 — TF-IDF (replace your existing cell)

# Named function instead of lambda — required for joblib pickling
def nepali_tokenizer(text):
    return text.split(' ')

tfidf = TfidfVectorizer(
    tokenizer    = nepali_tokenizer,   # ← named function, not lambda
    sublinear_tf = True,
    encoding     = 'utf-8',
    decode_error = 'ignore',
    ngram_range  = (1, 2),
    min_df       = 2,
    max_df       = 0.95,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_)}")
print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test  matrix shape: {X_test_tfidf.shape}")

TF-IDF vocab size : 196027
Train matrix shape: (71926, 196027)
Test  matrix shape: (17982, 196027)


## 7. Count Vectorizer (Trigram)

Used as an alternative feature set. Trigrams capture longer political phrases.

In [26]:
# Section 7 — Count Vectorizer (replace your existing cell)

cvec = CountVectorizer(
    tokenizer   = nepali_tokenizer,    # ← same named function
    ngram_range  = (1, 3),
    min_df       = 2,
    max_df       = 0.95,
)

X_train_cv = cvec.fit_transform(X_train)
X_test_cv  = cvec.transform(X_test)

print(f"Count Vectorizer vocab size : {len(cvec.vocabulary_)}")
print(f"Train matrix shape          : {X_train_cv.shape}")

Count Vectorizer vocab size : 324415
Train matrix shape          : (71926, 324415)


## 8. Evaluation Helper Function

In [27]:
results_store = {}   # accumulate all model results here

def evaluate(name, model, X_tr, y_tr, X_te, y_te, store=True):
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred    = model.predict(X_te)

    train_acc = accuracy_score(y_tr, y_pred_tr)
    test_acc  = accuracy_score(y_te, y_pred)
    prec      = precision_score(y_te, y_pred, zero_division=0)
    rec       = recall_score(y_te, y_pred, zero_division=0)
    f1        = f1_score(y_te, y_pred, zero_division=0)

    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Train Accuracy : {train_acc*100:.2f}%")
    print(f"  Test  Accuracy : {test_acc*100:.2f}%")
    print(f"  Precision      : {prec*100:.2f}%")
    print(f"  Recall         : {rec*100:.2f}%")
    print(f"  F1 Score       : {f1*100:.2f}%")
    print()
    print(classification_report(y_te, y_pred,
          target_names=['Positive(0)', 'Negative(1)']))

    # Confusion matrix plot
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred,
        display_labels=['Positive', 'Negative'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'{name}  —  Test Acc: {test_acc*100:.1f}%', fontsize=10)
    plt.tight_layout()
    plt.show()

    if store:
        results_store[name] = {
            'model'     : model,
            'train_acc' : round(train_acc*100, 2),
            'test_acc'  : round(test_acc*100, 2),
            'precision' : round(prec*100, 2),
            'recall'    : round(rec*100, 2),
            'f1'        : round(f1*100, 2),
        }
    return model


## 9. Train All Classifiers on TF-IDF Features

### 9a. Logistic Regression

In [28]:
lr = evaluate(
    'Logistic Regression (TF-IDF)',
    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_SEED),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)


  Logistic Regression (TF-IDF)
  Train Accuracy : 88.78%
  Test  Accuracy : 78.27%
  Precision      : 78.45%
  Recall         : 77.96%
  F1 Score       : 78.20%

              precision    recall  f1-score   support

 Positive(0)       0.78      0.79      0.78      8991
 Negative(1)       0.78      0.78      0.78      8991

    accuracy                           0.78     17982
   macro avg       0.78      0.78      0.78     17982
weighted avg       0.78      0.78      0.78     17982



### 9b. Linear SVM  ← typically best on text

In [29]:
svm_linear = evaluate(
    'Linear SVM (TF-IDF)',
    LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_SEED),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)


  Linear SVM (TF-IDF)
  Train Accuracy : 98.25%
  Test  Accuracy : 81.39%
  Precision      : 81.13%
  Recall         : 81.82%
  F1 Score       : 81.47%

              precision    recall  f1-score   support

 Positive(0)       0.82      0.81      0.81      8991
 Negative(1)       0.81      0.82      0.81      8991

    accuracy                           0.81     17982
   macro avg       0.81      0.81      0.81     17982
weighted avg       0.81      0.81      0.81     17982



### 9c. SVM — RBF kernel

### 9d. Random Forest

In [30]:
rf = evaluate(
    'Random Forest (TF-IDF)',
    RandomForestClassifier(n_estimators=200, criterion='entropy',
                           max_depth=None, random_state=RANDOM_SEED, n_jobs=-1),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)


  Random Forest (TF-IDF)
  Train Accuracy : 99.98%
  Test  Accuracy : 82.93%
  Precision      : 85.44%
  Recall         : 79.40%
  F1 Score       : 82.31%

              precision    recall  f1-score   support

 Positive(0)       0.81      0.86      0.84      8991
 Negative(1)       0.85      0.79      0.82      8991

    accuracy                           0.83     17982
   macro avg       0.83      0.83      0.83     17982
weighted avg       0.83      0.83      0.83     17982



### 9e. Decision Tree

In [31]:
dt = evaluate(
    'Decision Tree (TF-IDF)',
    DecisionTreeClassifier(criterion='entropy', max_depth=15, random_state=RANDOM_SEED),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)


  Decision Tree (TF-IDF)
  Train Accuracy : 67.21%
  Test  Accuracy : 64.04%
  Precision      : 61.45%
  Recall         : 75.33%
  F1 Score       : 67.69%

              precision    recall  f1-score   support

 Positive(0)       0.68      0.53      0.59      8991
 Negative(1)       0.61      0.75      0.68      8991

    accuracy                           0.64     17982
   macro avg       0.65      0.64      0.64     17982
weighted avg       0.65      0.64      0.64     17982



### 9f. Multinomial Naive Bayes

In [32]:
mnb = evaluate(
    'MultinomialNB (TF-IDF)',
    MultinomialNB(alpha=0.1),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)


  MultinomialNB (TF-IDF)
  Train Accuracy : 92.92%
  Test  Accuracy : 77.57%
  Precision      : 81.63%
  Recall         : 71.16%
  F1 Score       : 76.04%

              precision    recall  f1-score   support

 Positive(0)       0.74      0.84      0.79      8991
 Negative(1)       0.82      0.71      0.76      8991

    accuracy                           0.78     17982
   macro avg       0.78      0.78      0.77     17982
weighted avg       0.78      0.78      0.77     17982



### 9g. AdaBoost

In [33]:
ada = evaluate(
    'AdaBoost (TF-IDF)',
    AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_SEED),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)


  AdaBoost (TF-IDF)
  Train Accuracy : 59.65%
  Test  Accuracy : 58.41%
  Precision      : 55.58%
  Recall         : 83.79%
  F1 Score       : 66.83%

              precision    recall  f1-score   support

 Positive(0)       0.67      0.33      0.44      8991
 Negative(1)       0.56      0.84      0.67      8991

    accuracy                           0.58     17982
   macro avg       0.61      0.58      0.56     17982
weighted avg       0.61      0.58      0.56     17982



### 9h. XGBoost

In [34]:
if XGBOOST_AVAILABLE:
    xgb = evaluate(
        'XGBoost (TF-IDF)',
        XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
                      use_label_encoder=False, eval_metric='logloss',
                      random_state=RANDOM_SEED, verbosity=0),
        X_train_tfidf, y_train, X_test_tfidf, y_test
    )
else:
    print("XGBoost skipped — not installed")


  XGBoost (TF-IDF)
  Train Accuracy : 76.10%
  Test  Accuracy : 70.68%
  Precision      : 71.82%
  Recall         : 68.06%
  F1 Score       : 69.89%

              precision    recall  f1-score   support

 Positive(0)       0.70      0.73      0.71      8991
 Negative(1)       0.72      0.68      0.70      8991

    accuracy                           0.71     17982
   macro avg       0.71      0.71      0.71     17982
weighted avg       0.71      0.71      0.71     17982



## 10. Best Two Classifiers Re-run on Count Vectorizer Features

Logistic Regression and Linear SVM on trigram Count Vectorizer features — often adds a few extra accuracy points.

In [35]:
evaluate(
    'Logistic Regression (Count Vec)',
    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_SEED),
    X_train_cv, y_train, X_test_cv, y_test
)

evaluate(
    'Linear SVM (Count Vec)',
    LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_SEED),
    X_train_cv, y_train, X_test_cv, y_test
)


  Logistic Regression (Count Vec)
  Train Accuracy : 97.80%
  Test  Accuracy : 81.85%
  Precision      : 83.39%
  Recall         : 79.55%
  F1 Score       : 81.42%

              precision    recall  f1-score   support

 Positive(0)       0.80      0.84      0.82      8991
 Negative(1)       0.83      0.80      0.81      8991

    accuracy                           0.82     17982
   macro avg       0.82      0.82      0.82     17982
weighted avg       0.82      0.82      0.82     17982

  Linear SVM (Count Vec)
  Train Accuracy : 99.82%
  Test  Accuracy : 81.59%
  Precision      : 81.86%
  Recall         : 81.17%
  F1 Score       : 81.51%

              precision    recall  f1-score   support

 Positive(0)       0.81      0.82      0.82      8991
 Negative(1)       0.82      0.81      0.82      8991

    accuracy                           0.82     17982
   macro avg       0.82      0.82      0.82     17982
weighted avg       0.82      0.82      0.82     17982



,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


## 11. 5-Fold Cross-Validation on Top Models

More reliable than a single train/test split — especially on smaller datasets.

In [36]:
cv_models = {
    'LogReg (TF-IDF)': LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED),
    'LinSVM (TF-IDF)': LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_SEED),
    # Random Forest removed from CV — too slow on sparse TF-IDF
    # Already evaluated on train/test split above
}

print(f"{'Model':<25} {'Mean CV Acc':>12} {'Std':>8}")
print('-' * 48)
for name, clf in cv_models.items():
    scores = cross_val_score(
        clf, X_train_tfidf, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring='accuracy',
        n_jobs=1        # ← sequential, avoids memory thrashing
    )
    print(f"{name:<25} {scores.mean()*100:>11.2f}%  ±{scores.std()*100:.2f}%")

Model                      Mean CV Acc      Std
------------------------------------------------
LogReg (TF-IDF)                 79.96%  ±0.26%
LinSVM (TF-IDF)                 81.29%  ±0.15%


## 12. Model Comparison Summary

In [37]:
summary = pd.DataFrame([
    {
        'Model'         : name,
        'Train Acc (%)' : v['train_acc'],
        'Test Acc (%)'  : v['test_acc'],
        'Precision (%)' : v['precision'],
        'Recall (%)'    : v['recall'],
        'F1 (%)'        : v['f1'],
    }
    for name, v in results_store.items()
]).sort_values('Test Acc (%)', ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))


                          Model  Train Acc (%)  Test Acc (%)  Precision (%)  Recall (%)  F1 (%)
         Random Forest (TF-IDF)          99.98         82.93          85.44       79.40   82.31
Logistic Regression (Count Vec)          97.80         81.85          83.39       79.55   81.42
         Linear SVM (Count Vec)          99.82         81.59          81.86       81.17   81.51
            Linear SVM (TF-IDF)          98.25         81.39          81.13       81.82   81.47
   Logistic Regression (TF-IDF)          88.78         78.27          78.45       77.96   78.20
         MultinomialNB (TF-IDF)          92.92         77.57          81.63       71.16   76.04
               XGBoost (TF-IDF)          76.10         70.68          71.82       68.06   69.89
         Decision Tree (TF-IDF)          67.21         64.04          61.45       75.33   67.69
              AdaBoost (TF-IDF)          59.65         58.41          55.58       83.79   66.83


In [38]:
# Bar chart of test accuracy
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#10b981' if v['test_acc'] == summary['Test Acc (%)'].max()
          else '#60a5fa' for v in results_store.values()]
ax.barh(list(results_store.keys()), [v['test_acc'] for v in results_store.values()],
        color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — Test Accuracy', fontsize=14, fontweight='bold')
ax.axvline(x=85, color='red', linestyle='--', linewidth=1.2, label='85% target')
ax.legend()
for i, (name, v) in enumerate(results_store.items()):
    ax.text(v['test_acc'] + 0.3, i, f"{v['test_acc']}%", va='center', fontsize=9)
ax.set_xlim(50, 102)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 13. Identify & Save Best Model

In [39]:
best_name = summary.iloc[0]['Model']
best_info = results_store[best_name]

print(f"🏆 Best model : {best_name}")
print(f"   Test Acc  : {best_info['test_acc']}%")
print(f"   F1 Score  : {best_info['f1']}%")
print(f"   Precision : {best_info['precision']}%")
print(f"   Recall    : {best_info['recall']}%")


🏆 Best model : Random Forest (TF-IDF)
   Test Acc  : 82.93%
   F1 Score  : 82.31%
   Precision : 85.44%
   Recall    : 79.4%


In [ ]:
# Save the best model + its vectorizer as a bundle
save_dir = 'best_ml_model'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(best_info['model'], os.path.join(save_dir, 'classifier.pkl'))
joblib.dump(tfidf,              os.path.join(save_dir, 'tfidf_vectorizer.pkl'))
joblib.dump(cvec,               os.path.join(save_dir, 'count_vectorizer.pkl'))

# save label map
with open(os.path.join(save_dir, 'label_map.json'), 'w') as f:
    json.dump({'0': 'Positive', '1': 'Negative'}, f)

print(f"✅ Saved to '{save_dir}/'")
print(f"   classifier.pkl        — trained {best_name}")
print(f"   tfidf_vectorizer.pkl  — fitted TF-IDF")
print(f"   count_vectorizer.pkl  — fitted Count Vec")
print(f"   label_map.json        — {{0:Positive, 1:Negative}}")


✅ Saved to 'best_ml_model/'
   classifier.pkl        — trained Random Forest (TF-IDF)
   tfidf_vectorizer.pkl  — fitted TF-IDF
   count_vectorizer.pkl  — fitted Count Vec
   label_map.json        — {0:Positive, 1:Negative}


## 14. Quick Inference on New Text

In [41]:
def predict_sentiment(texts, classifier, vectorizer, label_map={0:'Positive', 1:'Negative'}):
    """
    texts      : list of pre-cleaned Nepali strings
    classifier : fitted sklearn model
    vectorizer : fitted TF-IDF or Count vectorizer
    """
    if isinstance(texts, str):
        texts = [texts]
    X = vectorizer.transform(texts)
    preds = classifier.predict(X)
    # get probabilities if available
    if hasattr(classifier, 'predict_proba'):
        probs = classifier.predict_proba(X)
        confs = probs.max(axis=1)
    elif hasattr(classifier, 'decision_function'):
        df_scores = classifier.decision_function(X)
        # convert distance to a 0-1 confidence
        confs = 1 / (1 + np.exp(-np.abs(df_scores)))
    else:
        confs = [None] * len(texts)

    for text, pred, conf in zip(texts, preds, confs):
        label = label_map[pred]
        conf_str = f'{conf*100:.1f}%' if conf is not None else 'N/A'
        print(f"  [{label:>8}  {conf_str}]  {text[:80]}")

# ── Example calls ────────────────────────────────────────────────
sample_texts = [
    'सरकारले राम्रो काम गरेको छ',
    'भ्रष्टाचारले देश सकियो',
    'नेताहरू जनताको सेवामा लागेका छन्',
    'यो सरकार पूर्ण रूपमा असफल भएको छ',
]

print("=== Predictions ===")
predict_sentiment(sample_texts, best_info['model'], tfidf)


=== Predictions ===
  [Negative  97.0%]  सरकारले राम्रो काम गरेको छ
  [Negative  74.5%]  भ्रष्टाचारले देश सकियो
  [Positive  73.0%]  नेताहरू जनताको सेवामा लागेका छन्
  [Positive  64.5%]  यो सरकार पूर्ण रूपमा असफल भएको छ
